# Reproduce the detection ablation

This notebook reproduces the two tables of the detection ablation (tile level on the test set and mosaic level
on the Diciembre orthomosaics) from the published checkpoints:

1. installs the environment used to produce the results (`requirements.txt`),
2. downloads the test tiles, the checkpoints and the orthomosaics (GitHub release `ablation-v1`),
3. evaluates the 16 detectors (8 architectures x 2 training sets) with their fixed, validation-selected
   score thresholds (`ablations/thresholds.json`),
4. builds the LaTeX tables and compares every value with the published ones.

**Runtime:** Google Colab with a GPU (*Runtime → Change runtime type → T4 GPU*), about 20–30 minutes. While the
repository is private, add a GitHub token with read access to it as the secret `GITHUB_TOKEN` (key icon on the
left); it is used to clone the repository and to download the release.

## 1. Code

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Mapping-rosettes" if IN_COLAB else os.getcwd()
if IN_COLAB and not os.path.exists(REPO):
    from google.colab import userdata
    try:
        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")  # also used to download the release data
    except Exception:
        pass
    token = os.environ.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/pauzca/Mapping-rosettes.git" if token else "https://github.com/pauzca/Mapping-rosettes.git"
    !git clone -q --branch reproduce {url} {REPO}
    assert os.path.exists(REPO), "could not clone the repository: add a GitHub token with read access as the GITHUB_TOKEN secret"
os.chdir(REPO)
assert os.path.exists("requirements.txt"), f"{REPO} is not the reproduce branch of Mapping-rosettes"
print("working in", os.getcwd())

## 2. Environment

Installs the package versions used for the results. pip may list conflicts with packages preinstalled in Colab
that this notebook does not use; they can be ignored. Colab's `torchaudio` is built for its own PyTorch and
breaks `transformers` once PyTorch is replaced, so it is removed (it is not needed).

In [ ]:
if IN_COLAB:
    !pip install -q -r {REPO}/requirements.txt
    !pip uninstall -q -y torchaudio
!python -c "import torch, ultralytics, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. Data

Downloads the test tiles, the checkpoints and the orthomosaics from the GitHub release `ablation-v1`
(~1.7 GB, checked against SHA-256 sums).

In [ ]:
DATA = os.environ.get("ROSETTES_DATA", "/content/data" if IN_COLAB else "data")
DATASETS = {"mixed": "mixed-resolution-frailejones", "lowres": "low-resolution-frailejones-detection"}
!python -m ablations.fetch --out {DATA}
ROOTS = {r: f"{DATA}/{slug}" for r, slug in DATASETS.items()}
assert all(os.path.exists(f"{root}/dataset/data.yaml") for root in ROOTS.values()), "data download failed"
print(ROOTS)

## 4. Evaluate the 16 detectors

Each model is evaluated on the test tiles (per source) and on the two Diciembre orthomosaics, at its fixed
validation-selected threshold. Results go to `outputs/eval/<regime>/<model>/`.

In [ ]:
import json
THRESHOLDS = json.load(open("ablations/thresholds.json"))
ULTRALYTICS = {"yolo11s", "yolov8s", "yolo11m", "rtdetr_l"}
MOSAICS = f"{ROOTS['mixed']}/mosaics"

for regime, models in THRESHOLDS.items():
    for model, thr in models.items():
        ckpt = f"{ROOTS[regime]}/checkpoints/{model}"
        spec = f"ultralytics:{ckpt}/weights/best.pt" if model in ULTRALYTICS else f"run:{ckpt}"
        print(f"=== {regime} / {model} (threshold {thr})", flush=True)
        !python -m ablations.evaluate_model --model {spec} --thr {thr} --data {ROOTS[regime]}/dataset/data.yaml \
            --mosaics-root {MOSAICS} --out outputs/eval/{regime}/{model} 2>&1 | grep -E "^(test all|dic)|Error"

## 5. Tables and comparison with the published values

Writes `outputs/eval/tabla_teselas.tex` and `outputs/eval/tabla_mosaicos.tex`, and reports how many values are
identical to the published tables and the largest difference (a different GPU or library build can flip a
detection right at a threshold; differences up to 0.005 are accepted).

In [ ]:
!python -m ablations.tables --eval-root outputs/eval
!python -m ablations.check --eval-root outputs/eval --tol 0.005